# FORESEE - Electrophilic ALP (Weak Violating)

### Load Libraries 

In [ ]:
import sys, os
src_path = "../../"
sys.path.append(src_path)
import numpy as np
from src.foresee import Foresee, Utility, Model
from matplotlib import pyplot as plt

## 1. Specifying the Model


The phenomenology of the leptophilic ALP can be described by the following Lagrangian

\begin{equation}
     \mathcal{L} = - \frac{1}{2} {m_{a}}^2 a^2  + \partial_{\mu} a\left(\frac{\bar{g}_{\ell \ell}}{2 m_{\ell}} \bar{\ell} \gamma^\mu \ell+\frac{g_{\ell \ell}}{2 m_{\ell}} \bar{\ell} \gamma^\mu \gamma_5 \ell+\frac{g_{\nu_{\ell}}}{2 m_{\ell}} \bar{\nu}_{\ell} \gamma^\mu P_L \nu_{\ell} \right)    
\end{equation}

with the ALP mass $m_a$ and the coupling $g_{\ell \ell}$ as free parameters. We focus here on the weak-violating (WV) electrophilic ALP scenario, where $\ell = e$ and  $g_{\ell\ell} = g_{ee}, \bar{g}_{\ell\ell} = g_{\nu_\ell} = 0$. For the search for ALPs at forward experiments we need to know i) the *production rate*, and ii) the *interaction rate*. All these properties are specified in the `Model` class. We initialize it with the name of the model as argument. 

In [ ]:
energy = "14"
modelname="ALP-e"
model = Model(modelname)

# Builder parameters, matching the build.py / load_model() defaults.
nsample_2body = 2000
nsample_3body = 2000
generators_light = ['EPOSLHC', 'SIBYLL', 'QGSJET'][:1]
generators_heavy = ['NLO-P8', 'NLO-P8-Max', 'NLO-P8-Min'][:1]

Electrophilic Axions get predominantly produced via hadron decay.  (for example Kaon decay into pions (FCNC) $K \to \pi^+ a$.)

To start, let us have a look at the kaon and B-meson spectra in terms of the angle with respect to the beam axis $\theta$ and the momentum $p$. This can be done using the function `get_spectrumplot` which requires the MC particle ID (or simply pid), the MC generator and the energy. The units on the coloraxis are pb/bin. 

FORESEE provides the 2D spectrum as tables for a variety of particles ($\pi^0$, $\eta$, ...), generators (SIBYLL, EPOSLHC, QGSJET, PYTHIA) and collision energies (14, 27 and 100 TeV). The datafiles are stored in the directory `files/hadrons`. 

**Production** The ALP is mainly produced in FCNC kaon and B-meson decays. The branching fractions are
\begin{equation}
    \text{BR}(K^+ \to \pi^+ a) = 45 \times g_{ee}^2 \times \lambda^{1/2}(m_{K}, m_\pi, m_a)
\end{equation}
\begin{equation}
\text{BR}(K_L \to \pi^0 a) = 27 \times g_{ee}^2 \times \lambda^{1/2}(m_{K^0}, m_{\pi^0}, m_a)
\end{equation}
\begin{equation}
\text{BR}(K_S \to \pi^0 a) = 0.3 \times g_{ee}^2 \times \lambda^{1/2}(m_{K^0}, m_{\pi^0}, m_a)
\end{equation}
\begin{equation}
\text{BR}(B \to X_s a)     = 1.6 \cdot10^5\times g_{ee}^2 \times \lambda^{1/2}(m_{B}, 0, m_a)
\end{equation}
where $\lambda$ is the Kallen function
\begin{equation}
\lambda(a,b,c) = \frac{a^4 + b^4 + c^4 - 2(a^2 b^2 + a^2c^2 + b^2c^2)}{a^4}.
\end{equation}
In the following, we model light hadron production using `EPOSLHC`, `SIBYLL` and `QGSJET` and heavy hadron production using the `POWHEG+Pythia8` predicions.

In [ ]:
model.add_production_2bodydecay(
    label="2body_321_211",
    pid0 = "321", # K_plus
    pid1 = "211", # pion
    br = "45 * coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body,
)

model.add_production_2bodydecay(
    label="2body_-321_-211",
    pid0 = "-321", # K_minus
    pid1 = "-211", # pion
    br = "45 * coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body,
)

model.add_production_2bodydecay(
    label="2body_130_111",
    pid0 = "130", # K_L
    pid1 = "111", # pi_0
    br = "27 *  coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body,
)

model.add_production_2bodydecay(
    label="2body_310_111",
    pid0 = "310", # K_S
    pid1 = "111", # pi_0
    br = "0.3 * coupling**2 * np.sqrt((1-(mass+self.masses('pid1'))**2/self.masses('pid0')**2)*(1-(mass-self.masses('pid1'))**2/self.masses('pid0')**2))",
    generator = generators_light,
    energy = energy,
    nsample = nsample_2body,
)

In [ ]:
model.add_production_2bodydecay(
    label="2body_511_130",
    pid0 = "511",
    pid1 = "130",
    br = "1.6e5 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    label="2body_-511_130",
    pid0 = "-511",
    pid1 = "130",
    br = "1.6e5 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    label="2body_521_321",
    pid0 = "521",
    pid1 = "321",
    br = "1.6e5 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    label="2body_-521_-321",
    pid0 = "-521",
    pid1 = "-321",
    br = "1.6e5 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    label="2body_531_333",
    pid0 = "531",
    pid1 = "333",
    br = "1.6e5 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
) 

model.add_production_2bodydecay(
    label="2body_-531_333",
    pid0 = "-531",
    pid1 = "333",
    br = "1.6e5 * coupling**2 * (1-(mass/self.masses('pid0'))**2)**2",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_2body, 
)  

Additionally, the ALPs can be produced through three body pseudoscalar meson decay $M\to e \nu_e a$ where $M = \pi^\pm, K^\pm, D^\pm, D_s^\pm$. The differential branching fraction for this processes are given by
\begin{equation}
\frac{dBR(\pi^\pm\to e\nu_e a)}{dE_a} = (7.6\cdot10^{6}\text{ GeV}^{-4})\times g_{ee}^2 \times (E_a^2 - m_a^2)^\frac{3}{2}
\end{equation}
\begin{equation}
\frac{dBR(K^\pm\to e\nu_e a)}{dE_a} = (9.9\cdot10^{4}\text{ GeV}^{-4})\times g_{ee}^2 \times (E_a^2 - m_a^2)^\frac{3}{2}
\end{equation}
\begin{equation}
\frac{dBR(D^\pm\to e\nu_e a)}{dE_a} = (5.5\cdot10^{2}\text{ GeV}^{-4})\times g_{ee}^2 \times (E_a^2 - m_a^2)^\frac{3}{2}
\end{equation}
\begin{equation}
\frac{dBR(D_s^\pm\to e\nu_e a)}{dE_a} = (7.9\cdot10^{3}\text{ GeV}^{-4})\times g_{ee}^2 \times (E_a^2 - m_a^2)^\frac{3}{2}
\end{equation}

In [ ]:
model.add_production_3bodydecay(
    label="3body_211_-11_-12",
    pid0 = "211",
    pid1 = "-11",
    pid2 = "-12",
    br = "7.6e6*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)

model.add_production_3bodydecay(
    label="3body_-211_11_12",
    pid0 = "-211",
    pid1 = "11",
    pid2 = "12",
    br = "7.6e6*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)

model.add_production_3bodydecay(
    label="3body_321_-11_-12",
    pid0 = "321",
    pid1 = "-11",
    pid2 = "-12",
    br = "9.9e4*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)

model.add_production_3bodydecay(
    label="3body_-321_11_12",
    pid0 = "-321",
    pid1 = "11",
    pid2 = "12",
    br = "9.9e4*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_light,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)

model.add_production_3bodydecay(
    label="3body_411_-11_-12",
    pid0 = "411",
    pid1 = "-11",
    pid2 = "-12",
    br = "5.5e2*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)

model.add_production_3bodydecay(
    label="3body_-411_11_12",
    pid0 = "-411",
    pid1 = "11",
    pid2 = "12",
    br = "5.5e2*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)
model.add_production_3bodydecay(
    label="3body_431_-11_-12",
    pid0 = "431",
    pid1 = "-11",
    pid2 = "-12",
    br = "7.9e3*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)

model.add_production_3bodydecay(
    label="3body_-431_11_12",
    pid0 = "-431",
    pid1 = "11",
    pid2 = "12",
    br = "7.9e3*coupling**2*(energy**2-mass**2)**(3/2)",
    generator = generators_heavy,
    energy = energy,
    nsample = nsample_3body,
    integration = "dE",
)


**Decay:** Electrophillic Alps at low masses decay primarily two electrons. 

In [ ]:
model.set_ctau_1d(
    filename="model/ctau_total.txt", 
    coupling_ref=1
)

branchings = [["e_e"     , "black"        , "solid" , r"$ee$"         , 0.110, 0.50],]
model.set_br_1d(
    modes=["e_e"],
    finalstates=[[11,-11]],
    filenames=["model/br/e_e.txt"]
)

In [ ]:
foresee = Foresee(path=src_path)
foresee.set_model(model=model)

## 3. Event Generation

In the following, we want to study one specific benchmark point with $m_{a}=10$ MeV and $g_{ee}= 1$ and export events as a HEPMC file. 

In [ ]:
mass, coupling, = 0.01, 1e-5

First, we will produce the corresponding flux for this mass and a reference coupling $g_{ref}=1$. 

In [ ]:
plot=foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the configuration of the detector (in terms of position, size and luminosity). Here we choose FASER during 2022/2023. 

In [ ]:
foresee.set_detector(
    distance=474, 
    selection="np.sqrt(x.x**2 + (x.y+0.065)**2)<.1", 
    length=4.0, 
    luminosity=60, 
)

For our benchmark point, let us now look at how many particle decay inside the decay volume. We also export 1000 unweighted events as a HEPMC file. 

In [ ]:
setupnames = ['POWHEG-central', 'POWHEG-max', 'POWHEG-min'][:1]

momenta, weights, _ = foresee.write_events(
    mass = mass, 
    coupling = coupling, 
    energy = energy, 
    numberevent = 1000,
    filename = "model/events/test.hepmc", 
    return_data = True,
    weightnames=setupnames,
    modes=None,
    nsample=10,
)

for isetup, setup in enumerate(setupnames):
    print("Expected number of events for "+setup+":", round(sum(weights[:,isetup]),3))

Let us plot the resulting energy distribution

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta], 
for isetup, setup in enumerate(setupnames):
    ax.hist(energies, weights=weights[:,isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup) 
ax.set_xscale("log")
ax.set_xlim(1e2,1e4) 
ax.set_xlabel("E [GeV]") 
ax.set_ylabel("Number of Events per Bin") 
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 3. Sensitivity Reach

In the following, we will obtain the projected sensitivity for the LLP model. For this, we first define a grid of couplings and masses, and then produce the corresponding fluxes. 

In [ ]:
masses=[round(x,5) for x in np.logspace(-2,np.log10(6.5),50)]
# Extra points around each production channel kinematic endpoint, from
# utility.production_thresholds(model, mass_range).
thresholds = [
    0.13489, 0.13906, 0.14323, 0.34762, 0.35837, 0.36912, 0.47837, 0.49317,
    0.50796, 1.81307, 1.86915, 1.9088, 1.92522, 1.96784, 2.02687, 4.21705,
    4.34747, 4.47789, 4.6404, 4.78392, 4.92744,
]
masses = sorted(masses + thresholds)
couplings = np.logspace(-8,-3,100) 

# Use cached LLP spectra: get_llp_spectrum recomputes on every call,
# so skip any masses already saved in model/LLP_spectra/.
for mass in masses:
    if not os.path.exists(f"model/LLP_spectra/{energy}TeV_m_{mass}.txt.gz"):
        foresee.get_llp_spectrum(mass=mass, coupling=1)

We can now plot the `production rate vs mass` using the `foresee.plot_production()` function. Below the production rates, we also show the decay branching fractions of the leading visible final states.

In [ ]:
productions=[
     {"channels": ['2body_130_111','2body_310_111','2body_321_211', '2body_-321_-211'], "color": "blue"   , "label": r"$K \to \pi a$"            , "generators": generators_light},
     {"channels": ['2body_511_130', '2body_-511_130', '2body_521_321', '2body_-521_-321'], "color": "red"    , "label": r"$B   \to X_s a$"  , "generators": generators_heavy},
     {"channels": ['2body_531_333', '2body_-531_333'], "color": "green"    , "label": r"$B_s^0   \to X_s a$"  , "generators": generators_heavy},
     {"channels": ['3body_211_-11_-12', '3body_-211_11_12'], "color": "orange"  , "label": r"$\pi \to e \nu_e a$"  , "generators": generators_light},
     {"channels": ['3body_321_-11_-12', '3body_-321_11_12'], "color": "purple"  , "label": r"$K \to e \nu_e a$"    , "generators": generators_light},
     {"channels": ['3body_411_-11_-12', '3body_-411_11_12'], "color": "brown"   , "label": r"$D \to e \nu_e a$"    , "generators": generators_heavy},
     {"channels": ['3body_431_-11_-12', '3body_-431_11_12'], "color": "magenta" , "label": r"$D_s \to e \nu_e a$"  , "generators": generators_heavy},
]
branchings = [
    ["e_e"  , "blue" , "solid", r"$e^+e^-$"    , 0.1, 0.6],
]

plot, ax, ax2 = foresee.plot_production(
    masses = masses,
    productions = productions,
    energy=energy,
    condition="logth<-3.7 and logp>2",  
    xlims=[0.01,10.0],ylims=[4e6,4e11],
    xlabel=r"Mass [GeV]",
    ylabel=r"Production Rate $\sigma/g^2$ [pb]",
    title=r"$\theta < 0.2$ mrad and $E > 100$ GeV",
    legendloc=(0.97,1),
    fs_label=12,
    ncol=2,
    figsize=(7,6),
    fs_label_br=9,
    branchings=branchings,
)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")


Let us now scan over various masses and couplings, and record the resulting number of events. Note that here we again consider the FASER configuration, which we set up before.

In [ ]:
setupnames = ['POWHEG-central']
modes = None

if energy == "13.6": detectors = [["FASER_R3"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 250 ,  None]]
elif energy == "14": detectors = [["FASER_HL"  , 480, "np.sqrt(x.x**2 + x.y**2)< .1", 1.5, 3000,  None], 
                                  ["FASER2_HL" , 650, "-1.5<x.x<1.5 and -.5<x.y<.5" , 10 , 3000,  None]]

condition = f"np.sqrt(p**2 + mass**2) > 100"

for detector in detectors: 

    #setup detector
    dlabel, distance, selection, length, luminosity, channels  = detector

    #skip detectors already precomputed (the plot cell reads them); scan only missing ones
    if all(os.path.exists(f"model/results/{energy}TeV_{dlabel}_{label}.npy") for label in setupnames):
        continue

    foresee.set_detector(distance=distance, selection=selection, length=length, luminosity=luminosity, channels=channels)

    #get reach  
    list_nevents = {label:[] for label in setupnames}
    for mass in masses:
        couplings, _, nevents, _, _  = foresee.get_events(mass=mass, energy=energy, couplings = couplings,modes=modes,nsample=10, preselectioncuts = condition)
        for i,label in enumerate(setupnames): list_nevents[label].append(nevents.T[i])  
            
    #save results
    configuration=dlabel
    for label in setupnames: 
        result = np.array([masses,couplings,list_nevents[label]], dtype='object')
        np.save("model/results/"+energy+"TeV_"+configuration+"_"+label+".npy",result)

Now let's plot the results. We first specify all production channels for which we want to show result (filename in model/results directory, label, color, linestyle, opacity alpha for filled contours, required number of events).

In [ ]:
setups = [ 
    ["13.6TeV_FASER_R3_POWHEG-central.npy"   , r"FASER (Run 3)"    , "firebrick"         ,  "solid"  , 0., 3],
    ["14TeV_FASER_HL_POWHEG-central.npy"   , r"FASER (HL-LHC)"    , "red"         ,  "dashed"  , 0., 3],
    ["14TeV_FASER2_HL_POWHEG-central.npy"   , r"FASER2 (HL-LHC)"    , "salmon"         ,  "dashed"  , 0., 3],    
]

Then we specify all the existing bounds (filename in model/bounds directory, label, label position x, label position y, label rotation)

In [ ]:
bounds = [   
    ["E137.txt",      "E137",  0.03, 4e-7, 0  ],
    ["NA62.txt",      "NA62",  0.025, 1.3e-6, 0  ],
    #["Konaka.txt",      "KEK",  0.0085, 5e-7, 90  ],
    ["PIONS.txt",      "SINDRUM",  0.045, 1.5e-5, 0  ],
    ["K_L.txt",      "E799",  0.18, 7.2*10**-6, 0  ],
    ["E865.txt",      "AGS",  0.16, 1.2e-5, 0  ],
    ["BMesons.txt",      "LHCb",  1.31e-2, 8e-5, -22  ],
    ["W.txt",    "LHC W decay"  ,  15e-3, 1.7e-4, 0  ], 
]

We then specify other projected sensitivitities (filename in model/bounds directory, color, label, label position x, label position y, label rotation)

In [ ]:
projections = [
    # ["Kaon_Projection.txt", "tomato",     "Kaon factorys",  1.4e-1, 3.6e-7, 0  ],
    # ["Pion_Projection.txt",   "dodgerblue",   "PIONEER",  1.07e-1, 4.1e-6, 0  ],
    # ["W_Decay_projection.txt",  "olive",     "LHC W decay",  1.3e-1, 2.6e-6,0 ],
]

Finally, we can plot everything using `foresee.plot_reach()`. 

In [ ]:
plot = foresee.plot_reach(
    setups=setups,
    bounds=bounds,
    projections=projections,
    title="Electrophillic ALP", 
    xlims = [0.01,2.0], 
    ylims=[4e-9,4e-4], 
    xlabel=r"ALP mass $m_{a}$ [GeV]", 
    ylabel=r"ALP coupling $g_{ee}$",
    legendloc=(1,0.2),
    linewidths=2,
)

os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()